# Tratamento do dataset de placas: 224×224, sem bordas brancas, cores preservadas

Lê o dataset (zip ou pasta, uma subpasta por classe), padroniza o enquadramento de **todas** as imagens e salva
uma cópia tratada em **JPG 224×224**, na mesma estrutura de pastas e com os mesmos nomes de arquivo.

**O que é feito em cada imagem**
1. leitura em **RGB** com a orientação EXIF corrigida;
2. remoção de faixas uniformes nas bordas (barras pretas/brancas que algumas fotos já têm);
3. **localização da placa** pela cor (azul ou cinza/preta) e **recorte quadrado** em volta dela, com uma folga;
   se a placa não for encontrada, **recorte quadrado central**;
4. fotos ou placas **muito largas/altas** (proporção acima de `WIDE_RATIO`): em vez de cortar as pontas (onde
   pode estar a sujeira), usa a maior região real possível e completa o quadrado com um **fundo desfocado da
   própria foto** — nunca branco, e sem duplicar a placa;
5. redimensionamento para **224×224** e gravação em **JPG qualidade 95, sem subamostragem de cor** (4:4:4).

**O que NÃO é feito:** nenhuma mudança de cor, brilho, contraste ou balanço de branco; nenhuma conversão para cinza.

**Saídas** (em `OUT_DIR`): a pasta tratada, um `.zip` dela, `relatorio_tratamento.csv` (o que foi feito em cada
imagem), uma prancha de conferência e o `preprocess.py` com o mesmo código, para aplicar igual na inferência.

## 1. Configuração

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

CFG = dict(
    INPUT       = "/content/Datasetv5.zip" if IN_COLAB else r"C:\Users\Aluno\Downloads\Datasetv5.zip",  # zip OU pasta
    OUT_DIR     = "/content/dataset_tratado" if IN_COLAB else r"C:\Users\Aluno\Downloads\dataset_tratado",
    OUT_NAME    = "Datasetv6_224",           # nome da pasta e do zip de saída

    IMG_SIZE    = 224,
    FRAMING     = "panel",                   # "panel" = recorta em volta da placa | "center" = recorte central
    FRAME_MARGIN= 0.06,                      # folga em volta da placa (fração do tamanho dela)
    CENTER_FRAC = 0.90,                      # recorte central: fração do lado menor
    WIDE_RATIO  = 1.5,                       # acima dessa proporção, completa com fundo desfocado em vez de cortar

    JPEG_QUALITY= 95,
    REMOVE_EXACT_DUPLICATES = True,          # mesma imagem repetida na mesma classe -> fica uma
    N_PREVIEW   = 8,                         # exemplos por classe na prancha de conferência
)
CFG

## 2. Imports e localização do dataset

In [ ]:
import hashlib, io, json, os, re, shutil, time, zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

S = CFG["IMG_SIZE"]
OUT = Path(CFG["OUT_DIR"]); OUT.mkdir(parents=True, exist_ok=True)
SRC = Path(CFG["INPUT"])

if not SRC.exists() and IN_COLAB:
    from google.colab import files
    print(f"{SRC} não encontrado. Selecione o zip do dataset:")
    up = files.upload()
    SRC = Path("/content") / next(iter(up))
assert SRC.exists(), f"não encontrado: {SRC}. Ajuste INPUT na célula 1."

if SRC.suffix.lower() == ".zip":
    RAW = OUT / "_original_extraido"
    if not RAW.exists():
        with zipfile.ZipFile(SRC) as z:
            z.extractall(RAW)
    print("zip extraído em", RAW)
else:
    RAW = SRC
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
arquivos = [p for p in RAW.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS and "__MACOSX" not in p.parts]
print(f"{len(arquivos)} imagens encontradas")
print(pd.Series([p.parent.name for p in arquivos]).value_counts().to_string())

## 3. Funções de tratamento

Este mesmo código é salvo como `preprocess.py`, para o backend (ou o script do ESP32) fazer exatamente o
mesmo com as fotos novas.

In [ ]:
PREPROCESS_CODE = r"""
import cv2
import numpy as np


def read_rgb(path):
    data = np.fromfile(str(path), dtype=np.uint8)        # funciona com acentos no caminho (Windows)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)            # aplica a orientação EXIF
    return None if img is None else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def trim_borders(img, tol=6.0, max_cut=0.2):
    # remove linhas/colunas uniformes nas bordas (barras de letterbox), até max_cut de cada lado
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    h, w = g.shape
    top, bottom, left, right = 0, h, 0, w
    while top < h * max_cut and g[top].std() < tol:
        top += 1
    while h - bottom < h * max_cut and g[bottom - 1].std() < tol:
        bottom -= 1
    while left < w * max_cut and g[:, left].std() < tol:
        left += 1
    while w - right < w * max_cut and g[:, right - 1].std() < tol:
        right -= 1
    return img[top:bottom, left:right]


def detect_panel_box(img, work=256, min_frac=0.12, max_frac=0.98, min_fill=0.35):
    # caixa (x0, y0, x1, y1) da placa pela cor; None se não encontrar ou se a detecção parecer duvidosa
    h, w = img.shape[:2]
    s = work / max(h, w)
    small = cv2.resize(img, (max(1, int(w * s)), max(1, int(h * s))), interpolation=cv2.INTER_AREA)
    hsv = cv2.cvtColor(small, cv2.COLOR_RGB2HSV).astype(np.int32)
    H, Sat, V = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    azul = (H >= 85) & (H <= 135) & (Sat >= 30) & (V >= 25)          # placas azuis (policristalinas / vidro)
    cinza = (Sat <= 40) & (V >= 55) & (V <= 190)                      # placas pretas/cinza (com reflexo ou poeira)
    mask = ((azul | cinza).astype(np.uint8)) * 255
    k = max(3, work // 40)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((k, k), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((k, k), np.uint8))
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n <= 1:
        return None, 0.0
    areas = stats[1:, cv2.CC_STAT_AREA]
    keep = [i + 1 for i, a in enumerate(areas) if a >= 0.15 * areas.max()]
    x0 = min(stats[i, cv2.CC_STAT_LEFT] for i in keep)
    y0 = min(stats[i, cv2.CC_STAT_TOP] for i in keep)
    x1 = max(stats[i, cv2.CC_STAT_LEFT] + stats[i, cv2.CC_STAT_WIDTH] for i in keep)
    y1 = max(stats[i, cv2.CC_STAT_TOP] + stats[i, cv2.CC_STAT_HEIGHT] for i in keep)
    frac = (x1 - x0) * (y1 - y0) / float(small.shape[0] * small.shape[1])
    if frac < min_frac or (mask[y0:y1, x0:x1] > 0).mean() < min_fill:
        return None, frac
    if frac > max_frac:                                               # a placa já ocupa a foto inteira
        return (0, 0, w, h), frac
    return (int(x0 / s), int(y0 / s), int(np.ceil(x1 / s)), int(np.ceil(y1 / s))), frac


def _clamp(a0, a1, lim):
    if a1 - a0 >= lim:
        return 0.0, float(lim)
    if a0 < 0:
        a1 -= a0; a0 = 0.0
    if a1 > lim:
        a0 -= a1 - lim; a1 = float(lim)
    return a0, a1


def blur_fill(part):
    # completa até quadrado com a PRÓPRIA imagem esticada e bem desfocada ao fundo (sem branco, sem duplicar a placa)
    ph, pw = part.shape[:2]
    side = max(ph, pw)
    bg = cv2.resize(part, (side, side), interpolation=cv2.INTER_AREA)
    k = max(3, (side // 8) | 1)
    bg = cv2.GaussianBlur(bg, (k, k), 0)
    y0, x0 = (side - ph) // 2, (side - pw) // 2
    bg[y0:y0 + ph, x0:x0 + pw] = part
    return bg


def square_region(img, box, margin, wide_ratio):
    # devolve (região quadrada, como foi feita). Recorta um quadrado em volta da caixa. Se a caixa for
    # alongada demais para caber num quadrado dentro da foto, usa a maior região real possível em volta dela
    # e completa o que faltar com um fundo desfocado da própria imagem (nada é cortado da placa).
    h, w = img.shape[:2]
    x0, y0, x1, y1 = [float(v) for v in box]
    bw, bh = x1 - x0, y1 - y0
    x0 -= bw * margin; x1 += bw * margin; y0 -= bh * margin; y1 += bh * margin
    x0, x1 = _clamp(x0, x1, w); y0, y1 = _clamp(y0, y1, h)
    bw, bh = x1 - x0, y1 - y0
    side = max(bw, bh)
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    if max(bw, bh) / max(min(bw, bh), 1) > wide_ratio and side > min(w, h):
        if bw >= bh:                                   # larga: estende para cima/baixo com conteúdo real
            y0, y1 = _clamp(cy - side / 2, cy + side / 2, h)
        else:                                          # alta: estende para os lados
            x0, x1 = _clamp(cx - side / 2, cx + side / 2, w)
        part = img[int(y0):int(round(y1)), int(x0):int(round(x1))]
        if part.shape[0] == part.shape[1]:
            return part, "recorte"
        return blur_fill(part), "fundo_desfocado"
    side = min(side, w, h)
    sx0, sx1 = _clamp(cx - side / 2, cx + side / 2, w)
    sy0, sy1 = _clamp(cy - side / 2, cy + side / 2, h)
    return img[int(sy0):int(round(sy1)), int(sx0):int(round(sx1))], "recorte"


def frame_image(img, size=IMG_SIZE, mode=FRAMING, margin=FRAME_MARGIN, center_frac=CENTER_FRAC, wide_ratio=WIDE_RATIO):
    # imagem RGB -> (imagem RGB size x size uint8, info). Use a MESMA função no treino e na inferência.
    img = trim_borders(img)
    h, w = img.shape[:2]
    info = {"deteccao": None, "frac_placa": None}
    box = None
    if mode == "panel":
        box, frac = detect_panel_box(img)
        info["frac_placa"] = round(float(frac), 3)
        info["deteccao"] = "placa" if box is not None else "nao_encontrada"
    if box is None:                                                   # recorte central (ou foto inteira, se for larga)
        if max(w, h) / min(w, h) > wide_ratio:
            box, margin = (0, 0, w, h), 0.0
        else:
            side = min(w, h) * center_frac
            box, margin = ((w - side) / 2, (h - side) / 2, (w + side) / 2, (h + side) / 2), 0.0
    part, how = square_region(img, box, margin, wide_ratio)
    info["metodo"] = ("placa" if info["deteccao"] == "placa" else "central") + "+" + how
    interp = cv2.INTER_AREA if max(part.shape[:2]) > size else cv2.INTER_CUBIC
    return cv2.resize(part, (size, size), interpolation=interp), info


def preprocess(path):
    # caminho da foto -> array float32 (1, size, size, 3) RGB 0..255
    img = read_rgb(path)
    if img is None:
        raise ValueError(f"não foi possível ler {path}")
    out, _ = frame_image(img)
    return out.astype("float32")[None]
"""

PREPROCESS_FILE = ('"""Pré-processamento das fotos de placas (gerado pelo notebook de tratamento).\n'
                   'Uso: from preprocess import read_rgb, frame_image; out, info = frame_image(read_rgb("foto.jpg"))\n"""\n'
                   f"IMG_SIZE = {S}\nFRAMING = {CFG['FRAMING']!r}\nFRAME_MARGIN = {CFG['FRAME_MARGIN']}\n"
                   f"CENTER_FRAC = {CFG['CENTER_FRAC']}\nWIDE_RATIO = {CFG['WIDE_RATIO']}\n" + PREPROCESS_CODE)
exec(compile(PREPROCESS_FILE, "preprocess.py", "exec"), globals())
(OUT / "preprocess.py").write_text(PREPROCESS_FILE, encoding="utf-8")
print("funções prontas | preprocess.py salvo em", OUT / "preprocess.py")

## 4. Tratar e salvar todas as imagens

Cada imagem vai para `OUT_DIR/OUT_NAME/<classe>/<mesmo nome>.jpg` (subpastas como `train/valid/test` são
mantidas). Arquivos ilegíveis e duplicatas exatas são listados no relatório.

In [ ]:
DST = OUT / CFG["OUT_NAME"]
if DST.exists():
    shutil.rmtree(DST)

def save_jpg(arr, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(arr).save(path, "JPEG", quality=CFG["JPEG_QUALITY"], subsampling=0, optimize=True)

linhas, vistos = [], {}
t0 = time.time()
for n, p in enumerate(sorted(arquivos), 1):
    rel = p.relative_to(RAW)
    classe = p.parent.name
    row = dict(origem=str(rel), classe=classe, destino=None, status=None, metodo=None, frac_placa=None,
               largura=None, altura=None)
    if CFG["REMOVE_EXACT_DUPLICATES"]:
        h = hashlib.md5(p.read_bytes()).hexdigest()
        if (h, classe) in vistos:
            row.update(status=f"duplicata de {vistos[(h, classe)]}")
            linhas.append(row); continue
        vistos[(h, classe)] = str(rel)
    img = read_rgb(p)
    if img is None:
        row.update(status="ilegivel"); linhas.append(row); continue
    out, info = frame_image(img)
    dest = (DST / rel).with_suffix(".jpg")
    if dest.exists():                                   # mesmo nome com extensões diferentes (ex.: a.png e a.jpg)
        dest = dest.with_name(dest.stem + "__" + p.suffix.lower().lstrip(".") + ".jpg")
    save_jpg(out, dest)
    row.update(destino=str(dest.relative_to(DST)), status="ok", metodo=info["metodo"], frac_placa=info["frac_placa"],
               altura=img.shape[0], largura=img.shape[1])
    linhas.append(row)
    if n % 200 == 0:
        print(f"  {n}/{len(arquivos)} ({time.time() - t0:.0f}s)")

rel_df = pd.DataFrame(linhas)
rel_df.to_csv(OUT / "relatorio_tratamento.csv", index=False, encoding="utf-8")
ok = rel_df[rel_df.status == "ok"]
print(f"\n{len(ok)} imagens salvas em {DST} ({time.time() - t0:.0f}s)")
print(rel_df.status.where(rel_df.status.isin(["ok", "ilegivel"]), "duplicata").value_counts().to_string())

## 5. Conferência

- **Tabela por classe:** se a placa for encontrada muito mais numa classe do que na outra (diferença acima de
  15 pontos), o tipo de recorte vira uma pista da classe para o modelo. Nesse caso, use `FRAMING="center"` e rode de novo.
- **Prancha:** original com a área usada (à esquerda) e o resultado 224×224 (à direita).

In [ ]:
tab = pd.crosstab(ok.metodo, ok.classe, normalize="columns").round(3) * 100
display(tab.rename_axis("método (% por classe)"))
if CFG["FRAMING"] == "panel":
    det = ok.metodo.str.startswith("placa").groupby(ok.classe).mean()
    print("placa encontrada por classe:", {c: f"{v:.0%}" for c, v in det.items()})
    if det.max() - det.min() > 0.15:
        print("ATENÇÃO: a detecção da placa varia mais de 15 pontos entre as classes. Considere FRAMING='center'.")
print(f"\nproporção original (largura/altura): mediana {(ok.largura / ok.altura).median():.2f} | "
      f"fotos com proporção > {CFG['WIDE_RATIO']}: {((ok.largura / ok.altura).clip(lower=1 / (ok.largura / ok.altura)) > CFG['WIDE_RATIO']).sum()}")

rng = np.random.default_rng(42)
classes = sorted(ok.classe.unique())
k = CFG["N_PREVIEW"]
fig, axes = plt.subplots(len(classes) * 2, k, figsize=(2.1 * k, 4.4 * len(classes)), squeeze=False)
for ci, c in enumerate(classes):
    sel = ok[ok.classe == c].sample(min(k, (ok.classe == c).sum()), random_state=1)
    for j, (_, r) in enumerate(sel.iterrows()):
        axes[2 * ci, j].imshow(trim_borders(read_rgb(RAW / r.origem)))
        axes[2 * ci, j].set_title(f"{c}\n{r.metodo}", fontsize=7)
        axes[2 * ci + 1, j].imshow(np.asarray(Image.open(DST / r.destino)))
        axes[2 * ci + 1, j].set_title(f"{S}x{S}", fontsize=7)
for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Original (sem as bordas uniformes) e resultado tratado", fontsize=10)
plt.tight_layout(); plt.savefig(OUT / "conferencia_tratamento.jpg", dpi=110); plt.show()

# checagem: todas 224x224 RGB
amostra = [DST / d for d in ok.destino.sample(min(50, len(ok)), random_state=0)]
tam = {Image.open(f).size for f in amostra}; modos = {Image.open(f).mode for f in amostra}
print("tamanhos:", tam, "| modos de cor:", modos)

## 6. Compactar e baixar

In [ ]:
zip_path = shutil.make_archive(str(OUT / CFG["OUT_NAME"]), "zip", root_dir=DST)
print("dataset tratado:", zip_path, f"({Path(zip_path).stat().st_size / 1e6:.1f} MB)")
print("relatório:", OUT / "relatorio_tratamento.csv")
print("preprocess.py:", OUT / "preprocess.py")
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
    files.download(str(OUT / "relatorio_tratamento.csv"))
    files.download(str(OUT / "preprocess.py"))

## 7. Como usar o dataset tratado nos notebooks de treino

- **Modelo grande (ResNet):** aponte `ZIP_PATH` para `Datasetv6_224.zip` e use `FRAMING = "none"` e
  `IMG_SIZE = 224` (as imagens já estão enquadradas; o notebook só confere o tamanho).
- **ESP32:** pode usar o mesmo zip; o notebook do ESP32 reduz de 224 para 112.
- **Na inferência**, aplique o `preprocess.py` deste notebook antes do modelo, para a foto nova passar pelo
  mesmo enquadramento.
- Os nomes dos arquivos foram mantidos (só a extensão vira `.jpg`), então o agrupamento por foto original
  (cópias do Roboflow, `synth__`) continua funcionando.